In [1]:
import pickle
import numpy as np
from scipy.sparse import csr_matrix, coo_matrix, dok_matrix
import scipy.sparse as sp
import torch as t
import torch.utils.data as data
import argparse
from torch.utils.data import Dataset, DataLoader
import os
import torch
from statistics import mean
from torch import nn
import torch.nn.functional as F
import sys
import time
import pickle
import pandas as pd
from tqdm import tqdm
import random
from sklearn import metrics
from sklearn.metrics import average_precision_score,auc,precision_recall_fscore_support
from torch.nn import Parameter
from argparse import ArgumentParser
from collections import Counter
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix, coo_matrix, vstack
from scipy.spatial.distance import pdist
from scipy import sparse
from torch.nn import Linear

In [2]:
dataset = "ml10m"
trnfile = "Data/" + dataset + "/trnMat.pkl"
with open(trnfile, 'rb') as fs:
    train_csr = (pickle.load(fs) != 0).astype(np.float32)

C:\Users\17721\AppData\Local\Temp\ipykernel_39332\3170330561.py:4: DeprecationWarning: Please import `coo_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.coo` namespace is deprecated and will be removed in SciPy 2.0.0.
  train_csr = (pickle.load(fs) != 0).astype(np.float32)


In [3]:
user_num, item_num = train_csr.shape

In [4]:
if type(train_csr) != coo_matrix:
    trnMat = sp.coo_matrix(train_csr)

In [5]:
trnMat

<69878x10195 sparse matrix of type '<class 'numpy.float32'>'
	with 6999171 stored elements in COOrdinate format>

In [6]:
def write_pkl(path, obj):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def build_item_item_graph_fast(trnMat, ui_k, sim_threshold, save_path):
    """
    输入:
        trnMat: scipy.sparse 矩阵 (用户-物品交互矩阵, COO/CSR都可)
        ui_k: 每个物品保留的 top-K 相似物品
        sim_threshold: 相似度阈值
    输出:
        保存物品–物品相似度图 (pkl)
    """

    # 转成CSR，方便矩阵乘法
    R = trnMat.tocsr()
    n_users, n_items = R.shape

    # 物品-物品共现矩阵 C = R^T R
    C = R.T @ R    # (n_items x n_items)

    # 每个物品的度
    item_deg = np.array(R.sum(axis=0)).flatten()

    # 转成CSR，方便逐行取邻居
    C = C.tocsr()

    final_row, final_col, final_val = [], [], []

    for i in tqdm(range(n_items), desc="build item-item graph"):
        start = C.indptr[i]
        end = C.indptr[i+1]
        neighs = C.indices[start:end]
        commons = C.data[start:end]

        if len(neighs) == 0:
            continue

        sims = []
        for j, inter in zip(neighs, commons):
            if i == j:
                continue
            union_size = item_deg[i] + item_deg[j] - inter
            if union_size == 0:
                continue
            sims.append((j, inter / union_size))

        if not sims:
            continue

        sims.sort(key=lambda x: x[1], reverse=True)

        # top-k
        for j, s in sims[:ui_k]:
            final_row.append(i)
            final_col.append(j)
            final_val.append(s)

        # 加入超过阈值的
        for j, s in sims:
            if s > sim_threshold and j not in [x[0] for x in sims[:ui_k]]:
                final_row.append(i)
                final_col.append(j)
                final_val.append(s)

    ii_mat = {'row': final_row, 'col': final_col, 'data': final_val}
    write_pkl(save_path, ii_mat)
    print(f"Item–item similarity graph saved to {save_path}")

In [7]:
ii_save_path = "Data/" + dataset + "/ii_graph.pkl"
build_item_item_graph_fast(trnMat, ui_k=10, sim_threshold=0.2, save_path=ii_save_path)

build item-item graph: 100%|█████████████████████████████████████████████████████| 10195/10195 [18:50<00:00,  9.02it/s]


Item–item similarity graph saved to Data/ml10m/ii_graph.pkl


In [8]:
ii_save_path = "Data/" + dataset + "/ii_graph.pkl"
ii_csr_save_path = "Data/" + dataset + "/ii10_csr"
with open(ii_save_path, "rb") as f:
    ii_simi_dict = pickle.load(f)
ii_rows, ii_cols, ii_data = ii_simi_dict["row"], ii_simi_dict["col"], ii_simi_dict["data"]
ii_csr_mat = sp.csr_matrix((ii_data, (ii_rows, ii_cols)), shape=(item_num, item_num))
with open(ii_csr_save_path, "wb") as f:
    pickle.dump(ii_csr_mat, f)